# Vector Index Generation

### This Notebook consists of:
 - Creation of a dense vector index in ElasticSearch for Vector Search
 - Bulk Insertions of the Dataset to the Index

In [ ]:
import os
import pandas as pd
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_core.documents import Document
import warnings
warnings.filterwarnings("ignore")
API_KEY = os.environ.get("ES_API_KEY")

In [ ]:
# Load the data
node_df = pd.read_parquet("../../03_data/preprocessed_data/nodes_beir_dbpedia.parquet")
# Create embedding model 
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/multi-qa-mpnet-base-cos-v1")
# Create dense vector store in Elasticsearch
elastic_vector_search = ElasticsearchStore(
    es_url="https://localhost:9200",
    index_name="dbpedia_vector_index_v3",
    embedding=embeddings,
    es_api_key=API_KEY,
    es_params={"verify_certs":False}
)

c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\elasticsearch\_sync\client\__init__.py:402: SecurityWarning: Connecting to 'https://localhost:9200' using TLS with verify_certs=False is insecure
  _transport = transport_class(
c:\Users\marvi\.conda\envs\masterthesis\Lib\site-packages\urllib3\connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'localhost'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [ ]:
# Create Langchain Documents and concatenate title and text as the field to vectorize
vector_ingest = []
for i, row in node_df.iterrows():
    vector_ingest.append(Document(
        page_content= row["title"] + " " + row["text"] if len(row["title"]) > 0 else row["text"],
        metadata={"dbpedia_id": str(row["id"])}))
ids = node_df["id"].to_list()

In [10]:
# Ingest vectors in batches of 10000 to reduce memory usage and GPU load
batch_size = 10000
for i in range(0, len(vector_ingest), batch_size):
    batch = vector_ingest[i:i + batch_size]
    batch_ids = ids[i:i + batch_size]
    elastic_vector_search.add_documents(documents=batch, ids=batch_ids)